# I) Solve an Equation Algebraically  (symbolically) 
There are two high-level functions to solve equations, solve() and solveset(). 

In [2]:
from sympy.abc import x, y
from sympy import solve
solve(x**2 - y, x, dict=True)

[{x: -sqrt(y)}, {x: sqrt(y)}]

In [5]:
from sympy import solveset
from sympy.abc import x, y
solveset(x**2 - y, x)

{-sqrt(y), sqrt(y)}

In [6]:
from sympy import Eq, solve, solveset
from sympy.abc import x, y
eqn = Eq(x**2, y)
eqn
solutions = solve(eqn, x, dict=True)
print(solutions)
solutions_set = solveset(eqn, x)
print(solutions_set)
for solution_set in solutions_set:
    print(solution_set)

[{x: -sqrt(y)}, {x: sqrt(y)}]
{-sqrt(y), sqrt(y)}
sqrt(y)
-sqrt(y)


Restrict the Domain of Solutions
By default, SymPy will return solutions in the complex domain, which also includes purely real and imaginary values. Here, the first two solutions are real, and the last two are imaginary:

In [ ]:
import sympy
from sympy import Symbol, solve, solveset
x = Symbol('x')
solve(x**4 - 256, x, dict=True)
solveset(x**4 - 256, x)

Restrict returned solutions to real numbers, or another domain or range, the different solving functions use different methods.

For solve(), place an assumption on the symbol to be solved for

In [1]:
from sympy import Symbol, solve
x = Symbol('x', real=True)
solve(x**4 - 256, x, dict=True)

[{x: -4}, {x: 4}]

or restrict the solutions with standard Python techniques for filtering a list such as a list comprehension:

In [1]:
from sympy import Or, Symbol, solve
x = Symbol('x', real=True)
expr = (x-4)*(x-3)*(x-2)*(x-1)
solution = solve(expr, x)
print(solution)
solution_outside_2_3 = [v for v in solution if (v.is_real and Or(v<2,v>3))]
print(solution_outside_2_3)

[1, 2, 3, 4]
[1, 4]


In [8]:
# For solveset(), limit the output domain in the function call by setting a domain

from sympy import S, solveset
from sympy.abc import x
solveset(x**4 - 256, x, domain=S.Reals)

# or by restricting returned solutions to any arbitrary set, including an interval:

from sympy import Interval, pi, sin, solveset
from sympy.abc import x
solveset(sin(x), x, Interval(-pi, pi))

{0, -pi, pi}

In [9]:
# if you restrict the solutions to a domain in which there are no solutions, solveset() will return the empty set, EmptySet:

from sympy import solveset, S
from sympy.abc import x
solveset(x**2 + 1, x, domain=S.Reals)

EmptySet


Explicitly Represent Infinite Sets of Possible Solutions
solveset() can represent infinite sets of possible solutions and express them in standard mathematical notation, for example  for every integer value of :

In [10]:

from sympy import pprint, sin, solveset
from sympy.abc import x
solution = solveset(sin(x), x)
pprint(solution)

{2⋅n⋅π │ n ∊ ℤ} ∪ {2⋅n⋅π + π │ n ∊ ℤ}


In [11]:

# However, solve() will return only a finite number of solutions: solve() tries to return just enough solutions so that all (infinitely many) solutions can generated from the returned solutions by adding integer multiples of the periodicity() of the equation

from sympy import sin, solve
from sympy.calculus.util import periodicity
from sympy.abc import x
f = sin(x)
solve(f, x)
periodicity(f, x)

2*pi

Use the Solution Result. Substitute Solutions From solve() Into an Expression
You can substitute solutions from solve() into an expression.

A common use case is finding the critical points and values for a function . At the critical points, the Derivative equals zero (or is undefined). You can then obtain the function values at those critical points by substituting the critical points back into the function using subs(). You can also tell if the critical point is a maxima or minima by substituting the values into the expression for the second derivative: a negative value indicates a maximum, and a positive value indicates a minimum.

In [12]:
from sympy.abc import x
from sympy import solve, diff
f = x**3 + x**2 - x
derivative = diff(f, x)
critical_points = solve(derivative, x, dict=True)
print(critical_points)
point1, point2 = critical_points
print(f.subs(point1))
print(f.subs(point2))
curvature = diff(f, x, 2)
print(curvature.subs(point1))
print(curvature.subs(point2))

[{x: -1}, {x: 1/3}]
1
-5/27
-4
4


solveset() Solution Sets Cannot Necessarily Be Interrogated Programmatically
If solveset() returns a finite set (class FiniteSet), you can iterate through the solutions:

In [13]:
from sympy import solveset
from sympy.abc import x, y
solution_set = solveset(x**2 - y, x)
print(solution_set)
solution_list = list(solution_set)
print(solution_list)

{-sqrt(y), sqrt(y)}
[sqrt(y), -sqrt(y)]


In [ ]:
# However, for more complex results, it may not be possible to list the solutions:

from sympy import S, solveset, symbols, intersection, sqrt
x, y = symbols('x, y')
solution_set = solveset(x**2 - y, x, domain=S.Reals)
print(solution_set)
intersection({-sqrt(y), sqrt(y)}, domain=S.Reals)
list(solution_set)

TypeError: The computation had not completed because of the undecidable set
membership is found in every candidates.

In this case, it is because, if  is negative, its square root would be imaginary rather than real and therefore outside the declared domain of the solution set. By declaring  to be real and positive, SymPy can determine that its square root is real, and thus resolve the intersection between the solutions and the set of real numbers:

In [15]:
from sympy import S, Symbol, solveset
x = Symbol('x')
y = Symbol('y', real=True, positive=True)
solution_set = solveset(x**2 - y, x, domain=S.Reals)
print(solution_set)

list(solution_set)

{-sqrt(y), sqrt(y)}


[sqrt(y), -sqrt(y)]

In [ ]:
# Alternatively, you can extract the sets from the solution set using args, then create a list from the set containing the symbolic solutions:

from sympy import S, solveset, symbols, intersection
x, y = symbols('x, y')
solution_set = solveset(x**2 - y, x, domain=S.Reals)
print(solution_set)
intersection({-sqrt(y), sqrt(y)}, domain=S.Reals)
solution_set_args = solution_set.args
print(solution_set.args)

list(solution_set_args[1])

Not All Equations Can Be Solved. Equations With No Closed-Form Solution
Some equations have no closed-form solution, in which case SymPy may return an empty set or give an error. For example, the following transcendental equation has no closed-form solution:

Equations Which Have a Closed-Form Solution, and SymPy Cannot Solve. 
It is also possible that there is an algebraic solution to your equation, and SymPy has not implemented an appropriate algorithm. 

In [ ]:
from sympy import cos, solve
from sympy.abc import x
solve(cos(x) - x, x, dict=True)